# parameter-subclass-of-tensor — ex2: trainable_params: filter module attrs by Parameter type, not MiniTensor

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parameter-subclass-of-tensor`. Running the final beacon cell reports progress against the `Backprop: Parameter subclasses Tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter subclasses Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-subclass-of-tensor`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-subclass-of-tensor"
DD_SUBTOPIC = "Backprop: Parameter subclasses Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Parameter — type as a runtime role signal — quick refresher

Ex1 established the IS-A relationship: `Parameter` subclasses `MiniTensor` so it passes `isinstance(_, MiniTensor)` checks in the wrapper layer. Ex2 uses the CONVERSE direction: `isinstance(_, Parameter)` distinguishes TRAINABLE state from intermediate tensors.

A real model has multiple kinds of Tensor-typed attributes:
- **Parameters** (weights, biases): trainable; the optimizer mutates them.
- **Buffers** (e.g. running mean in BatchNorm): MiniTensor but NOT Parameter; saved with the model but not updated by the optimizer.
- **Activations** / scratch tensors: incidental; not saved.

`isinstance(x, Parameter)` is the runtime gate that says 'this one gets gradient-descent updates'.

### Exercise 2 — trainable_params: filter module attrs by Parameter type, not MiniTensor

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Parameter-type filter to separate trainable params from buffers and incidental tensors: walk module attributes, yield only those that are `isinstance(_, Parameter)`.
> Keywords: parameter, trainable, buffer, filter, isinstance
> ```

**KCs targeted:** `parameter-subclass-of-tensor`, `get-children-callable-param`

We've given you the `Parameter` class from ex1 (subclass of MiniTensor, default `requires_grad=True`) and a `Module` base with `__dict__`-based attribute storage. Implement `trainable_params(module)` — a generator that yields each `(name, value)` where `isinstance(value, Parameter)` specifically.

It must distinguish:
- **Parameters** → YIELD (these are the trainables).
- **Plain MiniTensors** (e.g. buffers, running statistics, intermediate caches) → SKIP. They're MiniTensor-typed but not Parameter-typed.
- **Raw torch.Tensors** → SKIP (not even MiniTensor).
- **Non-tensor attrs** (ints, strings, layer config) → SKIP.

Signature: `trainable_params(module) -> generator of (name, Parameter)`.

**Why a separate function from ex1's `get_children`.** Ex1 used `isinstance(_, MiniTensor)` — that catches Parameters AND buffers AND any other MiniTensor-typed attrs. For an optimizer, we want STRICTLY trainable: Parameter only. `isinstance(_, Parameter)` is the strict-subset filter.

This is exactly the design split in `nn.Module`: `parameters()` and `buffers()` are two different walkers, distinguished by the registered type.

In [ ]:
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    """Tiny nn.Module stand-in."""

def trainable_params(module):
    """Yield (name, Parameter) for each attr that is_instance Parameter."""
    raise NotImplementedError()


def _test_ex2():
    # --- pure-Parameter module: all attrs yielded ---
    class Linear(Module):
        def __init__(self):
            self.weight = Parameter(t.randn(4, 3))
            self.bias = Parameter(t.zeros(4))
            self.in_features = 3
            self.out_features = 4

    lin = Linear()
    names = [n for n, _ in trainable_params(lin)]
    assert names == ['weight', 'bias'], f'pure-Parameter case: {names}'

    # --- mixed: Parameters + plain MiniTensors (buffers) → only Parameters yielded ---
    class BatchNorm(Module):
        def __init__(self):
            # trainable
            self.gamma = Parameter(t.ones(4))
            self.beta = Parameter(t.zeros(4))
            # buffers — MiniTensor but NOT Parameter (saved, but not optimized)
            self.running_mean = MiniTensor(t.zeros(4))
            self.running_var = MiniTensor(t.ones(4))

    bn = BatchNorm()
    names = [n for n, _ in trainable_params(bn)]
    assert names == ['gamma', 'beta'], (
        'plain MiniTensor (buffer) must NOT be yielded — only Parameter — '
        f'got {names}'
    )

    # --- raw torch.Tensor must NOT be yielded ---
    class Mixed(Module):
        def __init__(self):
            self.W = Parameter(t.randn(3, 3))
            self.cache = t.zeros(3)             # raw torch.Tensor — skip
            self.b = Parameter(t.zeros(3))

    mx = Mixed()
    names = [n for n, _ in trainable_params(mx)]
    assert names == ['W', 'b'], (
        'raw torch.Tensor must be skipped (not Parameter, not MiniTensor)'
    )

    # --- empty module: yields nothing, no crash ---
    class Empty(Module):
        def __init__(self):
            pass
    assert list(trainable_params(Empty())) == []

    # --- values are the actual Parameter instances (identity, not copies) ---
    p = next(trainable_params(lin))
    assert p[1] is lin.weight, 'value must BE the attribute, not a copy'

    # --- it is a generator (cheap iteration) ---
    import inspect
    iterator = trainable_params(lin)
    assert iter(iterator) is iterator or inspect.isgenerator(iterator), (
        'trainable_params must be a generator'
    )

    # --- Parameter with requires_grad=False (frozen) STILL counts as trainable_params ---
    # The type tag is what matters — requires_grad is mutable. A frozen layer is
    # still 'a Parameter that happens to be frozen', not 'a buffer'.
    class WithFrozen(Module):
        def __init__(self):
            self.W = Parameter(t.randn(3), requires_grad=False)  # frozen but still a Parameter
            self.b = Parameter(t.zeros(3))

    wf = WithFrozen()
    names = [n for n, _ in trainable_params(wf)]
    assert names == ['W', 'b'], (
        f'frozen Parameter (rg=False) still counts — filter on TYPE not rg, got {names}'
    )

    # --- isinstance(Parameter, MiniTensor) check: precondition for ex1's get_children ---
    p = Parameter(t.zeros(3))
    assert isinstance(p, MiniTensor), (
        'Parameter must still be a MiniTensor (ex1 invariant) — '
        'otherwise the wrapper helpers silently drop it'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class Parameter(MiniTensor):
    def __init__(self, array, requires_grad: bool = True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    pass

def trainable_params(module):
    for name, val in module.__dict__.items():
        if isinstance(val, Parameter):
            yield name, val
```

**Why filter on type, not on `requires_grad`.** A frozen Parameter has `requires_grad=False` but is STILL a Parameter (typed as trainable, just temporarily frozen). Filtering by `requires_grad` would skip frozen params — wrong for `state_dict` (we still want to save them) and for checkpointing.

**Why this is the converse of ex1's get_children.** `get_children` uses `isinstance(_, MiniTensor)` — the SUPERTYPE check, catches all wrapper-typed tensors. `trainable_params` uses `isinstance(_, Parameter)` — the SUBTYPE check, catches only the typed-trainable subset. Same `__dict__` walk, different filter.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()